
# \(d=3,\;Y4\) SU(3) vertex-singlet dictionary generator

This notebook builds the first missing microscopic data asset required to extend `ymcirc` to the certified fourth-order \(SU(3)\) calculation.

It generates the exact singlet multiplicity for every oriented six-link configuration at one cubic-lattice vertex using

\[
1,\;3,\;\bar3,\;6,\;\bar6,\;8,\;10,\;\overline{10},\;15,\;\overline{15}.
\]

There are \(10^6=1{,}000{,}000\) ordered local assignments. The notebook reduces the work to 1000 ordered triple decompositions using

\[
\dim\operatorname{Inv}\!\left(
R_{+x}\otimes R_{+y}\otimes R_{+z}
\otimes \bar R_{-x}\otimes\bar R_{-y}\otimes\bar R_{-z}
\right)
=
\sum_Q m_{\rm out}(Q)m_{\rm in}(Q).
\]

Use a **standard Colab CPU runtime**.


In [ ]:

from pathlib import Path
import subprocess, sys

ROOT = Path("/content")
REPO = ROOT / "pyclebsch_repo"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/hepqis-uiuc/pyclebsch.git", str(REPO)],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", "35e3926b07761da5bfbe573fd6af74e64a0c1a82"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "NOTE_MISC_requirements.txt")],
    check=True,
)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pyclebsch.su_n_operators as ops

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
assert commit == "35e3926b07761da5bfbe573fd6af74e64a0c1a82"
print("pyclebsch commit:", commit)


In [ ]:

from __future__ import annotations

import gzip
import itertools
import json
import math
import time
from collections import Counter
from pathlib import Path

import numpy as np

OUT = Path("/content/Y4_D3_VERTEX")
OUT.mkdir(parents=True, exist_ok=True)

IRREPS = [
    (0,0,0), (1,0,0), (1,1,0), (2,0,0), (2,2,0),
    (2,1,0), (3,0,0), (3,3,0), (3,1,0), (3,2,0),
]
IRREP_NAMES = [
    "1", "3", "3bar", "6", "6bar", "8",
    "10", "10bar", "15", "15bar",
]
IRREP_INDEX = {irrep: i for i, irrep in enumerate(IRREPS)}

def conjugate_iweight(iw):
    a, b, c = map(int, iw)
    assert c == 0 and a >= b >= 0
    return (a, a-b, 0)

for iw in IRREPS:
    assert conjugate_iweight(conjugate_iweight(iw)) == iw

assert conjugate_iweight((1,0,0)) == (1,1,0)
assert conjugate_iweight((2,0,0)) == (2,2,0)
assert conjugate_iweight((2,1,0)) == (2,1,0)
assert conjugate_iweight((3,1,0)) == (3,2,0)

TRIPLES = list(itertools.product(range(10), repeat=3))
assert len(TRIPLES) == 1000

def encode_triple(indices):
    a, b, c = map(int, indices)
    return 100*a + 10*b + c

def decode_triple(code):
    code = int(code)
    return (code // 100, (code // 10) % 10, code % 10)

for i, triple in enumerate(TRIPLES):
    assert encode_triple(triple) == i
    assert decode_triple(i) == triple

print("Irreps :", dict(enumerate(IRREP_NAMES)))
print("Triples:", len(TRIPLES))


## Verify the certified representation skeleton

In [ ]:

CERTIFIED_EDGES = json.loads('[{"edge_id":"FTI-01","source_dynkin":[0,0],"token":-1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-02","source_dynkin":[0,0],"token":1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-03","source_dynkin":[0,1],"token":-1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-04","source_dynkin":[0,1],"token":-1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-05","source_dynkin":[0,1],"token":1,"target_dynkin":[0,0],"target_dimension":1},{"edge_id":"FTI-06","source_dynkin":[0,1],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-07","source_dynkin":[0,2],"token":-1,"target_dynkin":[0,3],"target_dimension":10},{"edge_id":"FTI-08","source_dynkin":[0,2],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-09","source_dynkin":[0,2],"token":1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-10","source_dynkin":[0,2],"token":1,"target_dynkin":[1,2],"target_dimension":15},{"edge_id":"FTI-11","source_dynkin":[0,3],"token":1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-12","source_dynkin":[1,0],"token":-1,"target_dynkin":[0,0],"target_dimension":1},{"edge_id":"FTI-13","source_dynkin":[1,0],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-14","source_dynkin":[1,0],"token":1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-15","source_dynkin":[1,0],"token":1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-16","source_dynkin":[1,1],"token":-1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-17","source_dynkin":[1,1],"token":-1,"target_dynkin":[1,2],"target_dimension":15},{"edge_id":"FTI-18","source_dynkin":[1,1],"token":-1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-19","source_dynkin":[1,1],"token":1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-20","source_dynkin":[1,1],"token":1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-21","source_dynkin":[1,1],"token":1,"target_dynkin":[2,1],"target_dimension":15},{"edge_id":"FTI-22","source_dynkin":[1,2],"token":-1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-23","source_dynkin":[1,2],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-24","source_dynkin":[2,0],"token":-1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-25","source_dynkin":[2,0],"token":-1,"target_dynkin":[2,1],"target_dimension":15},{"edge_id":"FTI-26","source_dynkin":[2,0],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-27","source_dynkin":[2,0],"token":1,"target_dynkin":[3,0],"target_dimension":10},{"edge_id":"FTI-28","source_dynkin":[2,1],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-29","source_dynkin":[2,1],"token":1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-30","source_dynkin":[3,0],"token":-1,"target_dynkin":[2,0],"target_dimension":6}]')

def dynkin_to_iweight(dynkin):
    p, q = map(int, dynkin)
    return (p+q, q, 0)

edge_results = []
for row in CERTIFIED_EDGES:
    source = dynkin_to_iweight(row["source_dynkin"])
    token = (1,0,0) if int(row["token"]) == 1 else (1,1,0)
    target = dynkin_to_iweight(row["target_dynkin"])
    decomposition = ops.find_direct_sum([source, token])
    multiplicity = int(decomposition.get(target, 0))
    dimension = int(ops.calc_dimension(target))
    passed = multiplicity == 1 and dimension == int(row["target_dimension"])
    edge_results.append((row["edge_id"], passed, multiplicity, dimension))

failed = [row for row in edge_results if not row[1]]
assert not failed, failed
assert len(edge_results) == 30
print("EDGE GATE PASS: 30/30 certified fusion edges")


## Generate all 1000 ordered triple decompositions

In [ ]:

CACHE = OUT / "y4_triple_decompositions.json.gz"

def key_iweight(iw):
    return ",".join(map(str, iw))

def parse_iweight(text):
    return tuple(map(int, text.split(",")))

if CACHE.exists():
    with gzip.open(CACHE, "rt", encoding="utf-8") as handle:
        raw_cache = json.load(handle)
    triple_decompositions = [
        {parse_iweight(k): int(v) for k, v in record.items()}
        for record in raw_cache
    ]
    assert len(triple_decompositions) == 1000
    print("Loaded cached triple decompositions.")
else:
    triple_decompositions = []
    started = time.time()

    for index, triple in enumerate(TRIPLES):
        irreps = [IRREPS[i] for i in triple]
        decomposition = {
            tuple(k): int(v)
            for k, v in ops.find_direct_sum(irreps).items()
            if int(v) != 0
        }

        lhs = math.prod(int(ops.calc_dimension(irrep)) for irrep in irreps)
        rhs = sum(
            int(mult) * int(ops.calc_dimension(irrep))
            for irrep, mult in decomposition.items()
        )
        assert lhs == rhs, (triple, lhs, rhs)

        triple_decompositions.append(decomposition)

        if (index + 1) % 50 == 0:
            print(
                f"[triples] {index+1:4d}/1000 "
                f"elapsed={time.time()-started:.1f}s",
                flush=True,
            )

    serializable = [
        {key_iweight(k): int(v) for k, v in sorted(record.items())}
        for record in triple_decompositions
    ]
    with gzip.GzipFile(
        filename=str(CACHE),
        mode="wb",
        compresslevel=9,
        mtime=0,
    ) as handle:
        handle.write(
            json.dumps(serializable, separators=(",", ":"), sort_keys=True).encode()
        )

assert len(triple_decompositions) == 1000
print("TRIPLE GATE PASS: 1000/1000 decompositions with exact dimension conservation")


## Build the million-entry six-link singlet table

In [ ]:

ALL_CHANNELS = sorted({
    irrep
    for decomposition in triple_decompositions
    for irrep in decomposition
})
CHANNEL_INDEX = {irrep: i for i, irrep in enumerate(ALL_CHANNELS)}

M = np.zeros((1000, len(ALL_CHANNELS)), dtype=np.int16)
for triple_index, decomposition in enumerate(triple_decompositions):
    for irrep, multiplicity in decomposition.items():
        M[triple_index, CHANNEL_INDEX[irrep]] = int(multiplicity)

MULT = M @ M.T
assert MULT.shape == (1000, 1000)
assert np.all(MULT >= 0)

nonzero = int(np.count_nonzero(MULT))
max_mult = int(MULT.max())
hist = Counter(map(int, MULT.ravel()))

print("Vertex assignments:", MULT.size)
print("Nonzero singlet assignments:", nonzero)
print("Maximum singlet multiplicity:", max_mult)
print("Multiplicity histogram:", dict(sorted(hist.items())))


## Independent six-fold decomposition checks

In [ ]:

anchor_pairs = [
    (0, 0),
    (111, 111),
    (123, 456),
    (555, 555),
    (678, 876),
    (999, 999),
    (5, 500),
    (321, 123),
    (808, 80),
    (246, 642),
]

anchor_results = []
for out_code, in_code in anchor_pairs:
    out_triple = decode_triple(out_code)
    in_triple = decode_triple(in_code)

    product = [IRREPS[i] for i in out_triple] + [
        conjugate_iweight(IRREPS[i]) for i in in_triple
    ]
    direct = ops.find_direct_sum(product)
    direct_singlets = int(direct.get((0,0,0), 0))
    factorized = int(MULT[out_code, in_code])

    result = {
        "out_code": out_code,
        "in_code": in_code,
        "direct": direct_singlets,
        "factorized": factorized,
        "passed": direct_singlets == factorized,
    }
    anchor_results.append(result)
    assert result["passed"], result

print("SIX-FOLD GATE PASS:", len(anchor_results), "independent anchors")


## Write the microscopic vertex data asset

In [ ]:

npz_path = OUT / "y4_d3_vertex_multiplicity_table.npz"
np.savez_compressed(
    npz_path,
    multiplicities=MULT,
    triple_indices=np.asarray(TRIPLES, dtype=np.uint8),
    irrep_iweights=np.asarray(IRREPS, dtype=np.int16),
    channel_iweights=np.asarray(ALL_CHANNELS, dtype=np.int16),
)

summary = {
    "version": "2026-06-13-d3-y4-vertex-v1",
    "orientation_convention": {
        "outgoing": ["+x", "+y", "+z"],
        "incoming": ["-x", "-y", "-z"],
        "local_tensor_product": (
            "R(+x) tensor R(+y) tensor R(+z) tensor "
            "conj(R(-x)) tensor conj(R(-y)) tensor conj(R(-z))"
        ),
    },
    "irreps": [
        {"index": i, "name": IRREP_NAMES[i], "iweight": list(IRREPS[i])}
        for i in range(10)
    ],
    "ordered_triples": 1000,
    "ordered_six_link_assignments": int(MULT.size),
    "nonzero_singlet_assignments": nonzero,
    "maximum_singlet_multiplicity": max_mult,
    "multiplicity_histogram": {
        str(k): int(v) for k, v in sorted(hist.items())
    },
    "intermediate_irrep_channels": len(ALL_CHANNELS),
    "certified_fusion_edges_passed": 30,
    "six_fold_direct_anchors": anchor_results,
    "files": {
        "multiplicity_table": npz_path.name,
        "triple_decompositions": CACHE.name,
    },
    "passed": True,
}

summary_path = OUT / "y4_d3_vertex_multiplicity_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

lookup_source = """from pathlib import Path
import numpy as np

_ROOT = Path(__file__).resolve().parent
_DATA = np.load(_ROOT / "y4_d3_vertex_multiplicity_table.npz")
MULT = _DATA["multiplicities"]
IRREPS = [tuple(map(int, row)) for row in _DATA["irrep_iweights"]]
IRREP_INDEX = {irrep: i for i, irrep in enumerate(IRREPS)}

def encode_triple(indices):
    a, b, c = map(int, indices)
    return 100*a + 10*b + c

def singlet_multiplicity(outgoing_indices, incoming_indices):
    return int(MULT[
        encode_triple(outgoing_indices),
        encode_triple(incoming_indices),
    ])

def singlet_multiplicity_iweights(outgoing, incoming):
    return singlet_multiplicity(
        [IRREP_INDEX[tuple(x)] for x in outgoing],
        [IRREP_INDEX[tuple(x)] for x in incoming],
    )
"""
lookup_path = OUT / "y4_d3_vertex_lookup.py"
lookup_path.write_text(lookup_source)

loaded = np.load(npz_path)
assert np.array_equal(loaded["multiplicities"], MULT)
assert json.loads(summary_path.read_text())["passed"] is True

print("OUTPUT GATE PASS")
print("NPZ    :", npz_path)
print("SUMMARY:", summary_path)
print("LOOKUP :", lookup_path)
print("CACHE  :", CACHE)



## What this completes

This notebook completes the **local \(d=3\) Gauss-law multiplicity layer** for the ten-irrep \(Y4\) truncation.

The next generator uses this table to enumerate the four-vertex physical plaquette states with 16 control links, then attaches explicit Clebsch–Gordan basis tensors to the multiplicity indices.
